# 🌟 Vis-Sam-3D Suite — Free Dual GPU (Tesla T4) 3D Reconstruction Server

Transforms any single 2D photo into **photorealistic 3D models (`.glb`)** and **3D Gaussian Splats (`.ply`)** powered by **Meta AI SAM 3D Objects** and **AI Segmentation**.

---
### 📋 How to Connect to Vis-Sam-3D Web Studio:
1. In the right panel under **Notebook Options** -> **Accelerator**, select **GPU T4 x 2**.
2. Click **Run All** (or run cells 1 through 5 sequentially).
3. In **Cell 5**, copy your unique public **Cloudflare Tunnel URL** (`https://...trycloudflare.com`).
4. Paste the URL into your **Vis-Sam-3D Suite** Web Studio (`http://localhost:3000`).
5. Upload photos, use AI Auto-Segmentation or promptable clicks, and generate 3D models!

> 💡 **Always-Running Server**: Cell 5 runs continuously with live heartbeats to prevent Kaggle idle timeouts.


In [ ]:
# 1. Verify NVIDIA GPU & Runtime Environment
import os, sys, subprocess

print('=' * 65)
print('  [GPU VERIFICATION] SAM 3D SUITE HARDWARE CHECK')
print('=' * 65)

!nvidia-smi --query-gpu=name,memory.total,memory.free --format=csv

import torch
print(f'\nPyTorch Version: {torch.__version__}')
print(f'CUDA Available:  {torch.cuda.is_available()}')
if torch.cuda.is_available():
    count = torch.cuda.device_count()
    print(f'GPU Device Count: {count}')
    for i in range(count):
        props = torch.cuda.get_device_properties(i)
        print(f'  [GPU {i}] {props.name} ({round(props.total_memory/1e9, 2)} GB VRAM)')
else:
    print('⚠️ WARNING: No GPU detected! Enable GPU Accelerator in Notebook Settings.')


In [ ]:
# 2. Install Vis-Sam-3D Suite, SAM 3D Objects, Dependencies & Model Weights
import os, sys, glob, shutil, types

print('=' * 65)
print('  [STEP 2/4] INSTALLING DEPENDENCIES, PYTORCH3D & SAM 3D WEIGHTS')
print('=' * 65)

# Set runtime environment flags
os.environ['HF_TOKEN'] = os.environ.get('HF_TOKEN', '<YOUR_HUGGINGFACE_TOKEN>')
os.environ['HUGGING_FACE_HUB_TOKEN'] = os.environ['HF_TOKEN']
os.environ['LIDRA_SKIP_INIT'] = 'true'
os.environ['SPCONV_ALGO'] = 'native'
os.environ['CUMM_DISABLE_JIT'] = '1'
os.environ['SPCONV_DISABLE_JIT'] = '1'
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
if 'CUDA_HOME' not in os.environ:
    os.environ['CUDA_HOME'] = '/usr/local/cuda'
os.environ['CONDA_PREFIX'] = os.environ.get('CONDA_PREFIX', '/root/.local/share/mamba/envs/sam3d-objects')

# 1. Install core FastAPI, 3D libraries and scientific dependencies
!pip install -q fastapi uvicorn python-multipart trimesh pycloudflared pyquaternion scipy pillow matplotlib imageio imageio-ffmpeg huggingface_hub omegaconf hydra-core spconv-cu121 gsplat rembg opencv-python-headless loguru fvcore iopath open3d astor roma timm optree fasteners einops-exts plyfile xatlas onnxruntime pymeshfix pyvista igraph lightning psutil

# 2. Install Git dependencies (utils3d, MoGe geometry engine, and PyTorch3D transforms)
!pip install -q --no-deps git+https://github.com/EasternJournalist/utils3d.git
!pip install -q --no-deps git+https://github.com/microsoft/MoGe.git
!PYTORCH3D_NO_EXTENSION=1 pip install -q --no-deps --no-build-isolation git+https://github.com/facebookresearch/pytorch3d.git

# 3. Clone Meta SAM 3D Objects repository
REPO_DIR = '/kaggle/working/sam-3d-objects'
if not os.path.exists(REPO_DIR):
    print('Cloning Meta AI sam-3d-objects repository...')
    os.system('git clone https://github.com/facebookresearch/sam-3d-objects ' + REPO_DIR)
else:
    print(f'sam-3d-objects repository already exists at {REPO_DIR}')

# 4. Download HuggingFace Model Weights via snapshot_download
HF_DIR = os.path.join(REPO_DIR, 'checkpoints', 'hf')
pipeline_cfg1 = os.path.join(HF_DIR, 'pipeline.yaml')
pipeline_cfg2 = os.path.join(HF_DIR, 'checkpoints', 'pipeline.yaml')

if not os.path.exists(pipeline_cfg1) and not os.path.exists(pipeline_cfg2):
    print('Downloading SAM 3D Objects checkpoints from HuggingFace...')
    os.makedirs(HF_DIR, exist_ok=True)
    from huggingface_hub import snapshot_download
    snapshot_download(repo_id='facebook/sam-3d-objects', local_dir=HF_DIR, token=os.environ['HF_TOKEN'])
    print('Model weights downloaded successfully.')
else:
    print('Checkpoints already downloaded.')

# 5. Resolve nested checkpoints if present
nested_chk = os.path.join(HF_DIR, 'checkpoints')
if os.path.exists(nested_chk):
    for f in glob.glob(f'{nested_chk}/*'):
        target = os.path.join(HF_DIR, os.path.basename(f))
        if not os.path.exists(target):
            try:
                os.symlink(f, target)
            except Exception:
                pass

# 6. Apply Dynamic Stage-Wise Memory Offloader to inference_pipeline.py
pipeline_py = os.path.join(REPO_DIR, 'sam3d_objects', 'pipeline', 'inference_pipeline.py')
if os.path.exists(pipeline_py):
    with open(pipeline_py, 'r', encoding='utf-8') as f:
        pipe_code = f.read()

    # Stage 1 offload
    if 'ss_generator = self.models["ss_generator"]' in pipe_code and 'ss_generator.to(self.device)' not in pipe_code:
        pipe_code = pipe_code.replace(
            'ss_generator = self.models["ss_generator"]',
            'ss_generator = self.models["ss_generator"].to(self.device)\n        if "ss_decoder" in self.models and hasattr(self.models["ss_decoder"], "to"):\n            self.models["ss_decoder"].to(self.device)'
        )
    if 'ss_generator.inference_steps = prev_inference_steps\n        return return_dict' in pipe_code and 'self.models["ss_generator"].to("cpu")' not in pipe_code:
        pipe_code = pipe_code.replace(
            'ss_generator.inference_steps = prev_inference_steps\n        return return_dict',
            'ss_generator.inference_steps = prev_inference_steps\n        if "ss_generator" in self.models and hasattr(self.models["ss_generator"], "to"):\n            self.models["ss_generator"].to("cpu")\n        if "ss_decoder" in self.models and hasattr(self.models["ss_decoder"], "to"):\n            self.models["ss_decoder"].to("cpu")\n        torch.cuda.empty_cache()\n        return return_dict'
        )
    elif 'return coords' in pipe_code and 'self.models["ss_generator"].to("cpu")' not in pipe_code:
        pipe_code = pipe_code.replace(
            'return coords',
            'if "ss_generator" in self.models and hasattr(self.models["ss_generator"], "to"):\n            self.models["ss_generator"].to("cpu")\n        if "ss_decoder" in self.models and hasattr(self.models["ss_decoder"], "to"):\n            self.models["ss_decoder"].to("cpu")\n        torch.cuda.empty_cache()\n        return coords'
        )

    # Stage 2 offload
    if 'slat_generator = self.models["slat_generator"]' in pipe_code and 'slat_generator.to(self.device)' not in pipe_code:
        pipe_code = pipe_code.replace(
            'slat_generator = self.models["slat_generator"]',
            'slat_generator = self.models["slat_generator"].to(self.device)'
        )
    if 'return slat' in pipe_code and 'self.models["slat_generator"].to("cpu")' not in pipe_code:
        pipe_code = pipe_code.replace(
            'return slat',
            'if "slat_generator" in self.models and hasattr(self.models["slat_generator"], "to"):\n            self.models["slat_generator"].to("cpu")\n        torch.cuda.empty_cache()\n        return slat'
        )

    # Stage 3 offload
    if 'logger.info("Decoding sparse latent...")' in pipe_code and 'slat_decoder_mesh.to(self.device)' not in pipe_code:
        pipe_code = pipe_code.replace(
            'logger.info("Decoding sparse latent...")',
            'logger.info("Decoding sparse latent...")\n        for m_name in ["slat_decoder_mesh", "slat_decoder_gs", "slat_decoder_gs_4"]:\n            if m_name in self.models and hasattr(self.models[m_name], "to"):\n                self.models[m_name].to(self.device)'
        )
    if 'return ret' in pipe_code and 'self.models["slat_decoder_mesh"].to("cpu")' not in pipe_code:
        pipe_code = pipe_code.replace(
            'return ret',
            'for m_name in ["slat_decoder_mesh", "slat_decoder_gs", "slat_decoder_gs_4"]:\n            if m_name in self.models and hasattr(self.models[m_name], "to"):\n                self.models[m_name].to("cpu")\n        torch.cuda.empty_cache()\n        return ret'
        )
    if 'return outputs' in pipe_code and 'self.models["slat_decoder_mesh"].to("cpu")' not in pipe_code:
        pipe_code = pipe_code.replace(
            'return outputs',
            'for m_name in ["slat_decoder_mesh", "slat_decoder_gs", "slat_decoder_gs_4"]:\n            if m_name in self.models and hasattr(self.models[m_name], "to"):\n                self.models[m_name].to("cpu")\n        torch.cuda.empty_cache()\n        return outputs'
        )

    with open(pipeline_py, 'w', encoding='utf-8') as f:
        f.write(pipe_code)
    print('Applied dynamic stage-wise memory offloader to inference_pipeline.py')

# 7. Patch notebook/inference.py (CONDA_PREFIX safeguard & pure torch quaternion)
inf_path = os.path.join(REPO_DIR, 'notebook', 'inference.py')
if os.path.exists(inf_path):
    with open(inf_path, 'r', encoding='utf-8') as f:
        inf_code = f.read()
    if 'os.environ["CUDA_HOME"] = os.environ["CONDA_PREFIX"]' in inf_code:
        inf_code = inf_code.replace(
            'os.environ["CUDA_HOME"] = os.environ["CONDA_PREFIX"]',
            'os.environ["CUDA_HOME"] = os.environ.get("CONDA_PREFIX", "/usr/local/cuda")'
        )
    # Remove unused Kaolin visualizer imports
    inf_code = inf_code.replace('from kaolin.visualize import IpyTurntableVisualizer\n', '# from kaolin.visualize import IpyTurntableVisualizer\n')
    inf_code = inf_code.replace('from kaolin.render.camera import Camera, CameraType\n', '# from kaolin.render.camera import Camera, CameraType\n')
    inf_code = inf_code.replace('from kaolin.render.camera import Camera, CameraExtrinsics, PinholeIntrinsics\n', '# from kaolin.render.camera import Camera, CameraExtrinsics, PinholeIntrinsics\n')
    # Pure PyTorch tensor quaternion fallback
    q_math = '''def quaternion_invert(q):
    return q * torch.tensor([1, -1, -1, -1], device=q.device, dtype=q.dtype)
def quaternion_multiply(a, b):
    aw, ax, ay, az = a[..., 0], a[..., 1], a[..., 2], a[..., 3]
    bw, bx, by, bz = b[..., 0], b[..., 1], b[..., 2], b[..., 3]
    return torch.stack((aw*bw-ax*bx-ay*by-az*bz, aw*bx+ax*bw+ay*bz-az*by, aw*by-ax*bz+ay*bw+az*bx, aw*bz+ax*by-ay*bx+az*bw), -1)
'''
    if 'from pytorch3d.transforms import quaternion_invert, quaternion_multiply' in inf_code:
        inf_code = inf_code.replace('from pytorch3d.transforms import quaternion_invert, quaternion_multiply', q_math)
    with open(inf_path, 'w', encoding='utf-8') as f:
        f.write(inf_code)
    shutil.copy2(inf_path, '/kaggle/working/inference.py')
    print('Patched inference.py and synced to /kaggle/working/inference.py')

# 8. Register sam3d_objects system-wide in editable mode
!pip install -e /kaggle/working/sam-3d-objects --no-deps

# 9. Add repo and notebook paths to sys.path
for p in ['/kaggle/working', os.path.join(REPO_DIR, 'notebook'), REPO_DIR]:
    if p not in sys.path:
        sys.path.insert(0, p)

print('\n✅ Step 2 Complete: Environment, SAM 3D Objects, Rembg & Weights Ready!')

# 10. Pre-warm high-fidelity background removal model (prevents table plane bleeding)
try:
    from rembg import new_session
    new_session("u2net")
    print('u2net background removal model pre-warmed successfully.')
except Exception as e:
    print(f'Rembg warm-up note: {e}')


In [ ]:
# @title ⚙️ Vis-Sam-3D Suite Server Configuration { display-mode: "form" }
SERVER_PORT = 8000  # @param {type: "integer"}
DEFAULT_INFERENCE_STEPS = 25  # @param {type: "integer"}
USE_CLOUDFLARE_TUNNEL = True  # @param {type: "boolean"}
ENABLE_WATERTIGHT_POSTPROCESS = True  # @param {type: "boolean"}

os.environ['SERVER_PORT'] = str(SERVER_PORT)
os.environ['SAM3D_STEPS'] = str(DEFAULT_INFERENCE_STEPS)
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
os.environ['SPCONV_ALGO'] = 'native'
os.environ['CUMM_DISABLE_JIT'] = '1'
os.environ['SPCONV_DISABLE_JIT'] = '1'
print('✅ Server configuration applied successfully.')


In [ ]:
# 3. Deploy SAM 3D Suite FastAPI Production Server
SERVER_CODE = '''# Inlined Production Server for Vis-Sam-3D Suite
import os, sys, time, json, uuid, asyncio, threading, glob, types, io, base64
from pathlib import Path
from typing import Optional, List, Dict, Any
import numpy as np
from PIL import Image
import torch
import trimesh
from fastapi import FastAPI, UploadFile, File, Form, HTTPException, BackgroundTasks
from fastapi.middleware.cors import CORSMiddleware
from fastapi.responses import JSONResponse, FileResponse, HTMLResponse, StreamingResponse
import uvicorn

# Runtime environment flags
os.environ["CUMM_DISABLE_JIT"] = "1"
os.environ["SPCONV_DISABLE_JIT"] = "1"
os.environ["LIDRA_SKIP_INIT"] = "true"
os.environ["SPCONV_ALGO"] = "native"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
if "CUDA_HOME" not in os.environ:
    os.environ["CUDA_HOME"] = "/usr/local/cuda"
os.environ["CONDA_PREFIX"] = os.environ.get("CONDA_PREFIX", "/root/.local/share/mamba/envs/sam3d-objects")

# 1. PyTorch3D C++ transform mock
if "pytorch3d._C" not in sys.modules:
    p3d_c = types.ModuleType("pytorch3d._C")
    p3d_c.MAX_UINT = 2**32 - 1
    p3d_c.EPS = 1e-5
    p3d_c.pulsar = types.ModuleType("pytorch3d._C.pulsar")
    p3d_c.pulsar.Renderer = type("Renderer", (), {})
    sys.modules["pytorch3d._C"] = p3d_c
    sys.modules["pytorch3d._C.pulsar"] = p3d_c.pulsar
    try:
        import pytorch3d
        pytorch3d._C = p3d_c
    except Exception:
        pass

# 2. Kaolin complete package mock with check_tensor validation
if "kaolin" not in sys.modules:
    kaolin = types.ModuleType("kaolin")
    kaolin.__path__ = []
    k_render = types.ModuleType("kaolin.render")
    k_render.__path__ = []
    k_cam = types.ModuleType("kaolin.render.camera")
    k_cam.Camera = type("Camera", (), {})
    k_cam.CameraType = type("CameraType", (), {})
    k_cam.CameraExtrinsics = type("CameraExtrinsics", (), {})
    k_cam.PinholeIntrinsics = type("PinholeIntrinsics", (), {})
    k_vis = types.ModuleType("kaolin.visualize")
    k_vis.IpyTurntableVisualizer = type("IpyTurntableVisualizer", (), {})
    k_utils = types.ModuleType("kaolin.utils")
    k_utils.__path__ = []
    k_testing = types.ModuleType("kaolin.utils.testing")
    k_testing.check_tensor = lambda *args, **kwargs: True
    k_utils.testing = k_testing
    kaolin.render = k_render
    kaolin.render.camera = k_cam
    kaolin.visualize = k_vis
    kaolin.utils = k_utils
    sys.modules["kaolin"] = kaolin
    sys.modules["kaolin.render"] = k_render
    sys.modules["kaolin.render.camera"] = k_cam
    sys.modules["kaolin.visualize"] = k_vis
    sys.modules["kaolin.utils"] = k_utils
    sys.modules["kaolin.utils.testing"] = k_testing

# 3. utils3d compatibility shims
try:
    import utils3d.numpy, utils3d.torch
    utils3d.numpy.depth_edge = getattr(utils3d.numpy, "depth_map_edge", getattr(utils3d.numpy, "depth_edge", None))
    utils3d.numpy.normals_edge = getattr(utils3d.numpy, "normal_map_edge", getattr(utils3d.numpy, "normals_edge", None))
    utils3d.numpy.points_to_normals = getattr(utils3d.numpy, "point_map_to_normal_map", getattr(utils3d.numpy, "points_to_normals", None))
    utils3d.numpy.image_uv = getattr(utils3d.numpy, "uv_map", getattr(utils3d.numpy, "image_uv", None))
    utils3d.numpy.image_mesh = getattr(utils3d.numpy, "build_mesh_from_map", getattr(utils3d.numpy, "image_mesh", None))
    utils3d.torch.depth_edge = getattr(utils3d.torch, "depth_map_edge", getattr(utils3d.torch, "depth_edge", None))
    utils3d.torch.points_to_normals = getattr(utils3d.torch, "point_map_to_normal_map", getattr(utils3d.torch, "points_to_normals", None))
    utils3d.torch.image_uv = getattr(utils3d.torch, "uv_map", getattr(utils3d.torch, "image_uv", None))
    utils3d.torch.image_mesh = getattr(utils3d.torch, "build_mesh_from_map", getattr(utils3d.torch, "image_mesh", None))
    if not hasattr(utils3d.torch, "perspective_from_fov_xy"):
        utils3d.torch.perspective_from_fov_xy = lambda fov_x, fov_y, near, far: utils3d.torch.perspective_from_fov(fov_x=fov_x, fov_y=fov_y, near=near, far=far)
    if not hasattr(utils3d.torch, "intrinsics_from_fov_xy"):
        utils3d.torch.intrinsics_from_fov_xy = lambda fov_x, fov_y: utils3d.torch.intrinsics_from_fov(fov_x=fov_x, fov_y=fov_y)
except Exception as e:
    print(f"utils3d warning: {e}")

WORKSPACE = "/kaggle/working"
REPO_DIR = os.path.join(WORKSPACE, "sam-3d-objects")
for p in [WORKSPACE, os.path.join(REPO_DIR, "notebook"), REPO_DIR]:
    if p not in sys.path:
        sys.path.insert(0, p)

BASE_DIR = Path("/kaggle/working/sam3d_storage")
UPLOADS_DIR = BASE_DIR / "uploads"
MODELS_DIR = BASE_DIR / "models"
MASKS_DIR = BASE_DIR / "masks"
for d in [UPLOADS_DIR, MODELS_DIR, MASKS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

app = FastAPI(title="Vis-Sam-3D Suite Server", version="1.0.0")
app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)

JOBS: Dict[str, Dict[str, Any]] = {}
PIPELINE_INSTANCE = None
PIPELINE_LOCK = threading.Lock()
CURRENT_TUNNEL_URL = ""

def get_pipeline():
    global PIPELINE_INSTANCE
    with PIPELINE_LOCK:
        if PIPELINE_INSTANCE is None:
            for p in [WORKSPACE, os.path.join(REPO_DIR, "notebook"), REPO_DIR]:
                if p not in sys.path:
                    sys.path.insert(0, p)
            from inference import Inference
            cfg_path = os.path.join(REPO_DIR, "checkpoints", "hf", "pipeline.yaml")
            if not os.path.exists(cfg_path):
                cfg_path = os.path.join(REPO_DIR, "checkpoints", "hf", "checkpoints", "pipeline.yaml")
            print(f"Loading SAM 3D Objects pipeline from {cfg_path}...")
            PIPELINE_INSTANCE = Inference(cfg_path, compile=False)
            pipe = PIPELINE_INSTANCE._pipeline
            for m_name in ["ss_generator", "slat_generator", "ss_decoder", "slat_decoder_mesh", "slat_decoder_gs", "slat_decoder_gs_4"]:
                if m_name in pipe.models and hasattr(pipe.models[m_name], "to"):
                    pipe.models[m_name].to("cpu")
            torch.cuda.empty_cache()
            print("SAM 3D Objects pipeline loaded with dynamic memory offloading!")
        return PIPELINE_INSTANCE

REMBG_SESSION = None

def get_rembg_session():
    global REMBG_SESSION
    if REMBG_SESSION is None:
        try:
            from rembg import new_session
            for model_name in ["u2net", "isnet-general-use"]:
                try:
                    REMBG_SESSION = new_session(model_name)
                    print(f"Initialized rembg session with {model_name}")
                    break
                except Exception:
                    continue
        except Exception as e:
            print(f"Could not initialize rembg session: {e}")
    return REMBG_SESSION

def auto_segment_image(pil_img: Image.Image, points: list = None, bbox: list = None) -> Image.Image:
    try:
        from PIL import ImageOps
        pil_img = ImageOps.exif_transpose(pil_img)
    except Exception:
        pass
    try:
        from rembg import remove
        session = get_rembg_session()
        if session is not None:
            mask_rem = remove(pil_img, session=session, only_mask=True)
        else:
            mask_rem = remove(pil_img, only_mask=True)
        mask_rem_np = np.array(mask_rem)
        if np.count_nonzero(mask_rem_np > 128) > 100:
            binary = (mask_rem_np > 128).astype(np.uint8) * 255
            if bbox and len(bbox) == 4:
                import cv2
                h, w = binary.shape[:2]
                x1, y1, x2, y2 = [int(v) for v in bbox]
                bbox_mask = np.zeros_like(binary)
                bbox_mask[max(0, y1):min(h, y2), max(0, x1):min(w, x2)] = 255
                binary = cv2.bitwise_and(binary, bbox_mask)
            return Image.fromarray(binary, mode="L")
    except Exception as e:
        print(f"Rembg segment error: {e}")
        pass

    w, h = pil_img.size
    import cv2
    try:
        img_cv = cv2.cvtColor(np.array(pil_img), cv2.COLOR_RGB2BGR)
        mask_cv = np.zeros((h, w), dtype=np.uint8)
        rect = (int(w * 0.10), int(h * 0.10), int(w * 0.80), int(h * 0.80))
        bgd = np.zeros((1, 65), np.float64)
        fgd = np.zeros((1, 65), np.float64)
        cv2.grabCut(img_cv, mask_cv, rect, bgd, fgd, 3, cv2.GC_INIT_WITH_RECT)
        final_mask = np.where((mask_cv == 2) | (mask_cv == 0), 0, 255).astype(np.uint8)
        if np.count_nonzero(final_mask) > 100:
            return Image.fromarray(final_mask, mode="L")
    except Exception:
        pass

    np_mask = np.zeros((h, w), dtype=np.uint8)
    cv2.ellipse(np_mask, (w // 2, h // 2), (int(w * 0.38), int(h * 0.38)), 0, 0, 360, 255, -1)
    return Image.fromarray(np_mask, mode="L")

RECON_LOCK = threading.Lock()

def worker(job_id: str, img_p: Path, mask_p: Optional[Path], name: str):
    job = JOBS[job_id]
    job["status"] = "processing"
    job["stage"] = "Diffusion & Latent Sampling"
    job["progress"] = 0.25
    job["started_at"] = time.time()
    t0 = time.time()
    try:
        inf = get_pipeline()
        img = Image.open(img_p).convert("RGB")
        try:
            from PIL import ImageOps
            img = ImageOps.exif_transpose(img)
        except Exception:
            pass
        img_np = np.array(img)
        
        if mask_p and mask_p.exists():
            mask_img = Image.open(mask_p)
            try:
                from PIL import ImageOps
                mask_img = ImageOps.exif_transpose(mask_img)
            except Exception:
                pass
            raw = np.array(mask_img)
            if raw.ndim == 2:
                mask_np = raw > 128
            elif raw.ndim == 3:
                # If 4-channel RGBA with true alpha variation (transparent background)
                if raw.shape[2] == 4 and (np.min(raw[..., 3]) < 200 and np.max(raw[..., 3]) > 50):
                    mask_np = raw[..., 3] > 128
                else:
                    # Black & white mask: white foreground (255), black background (0)
                    mask_np = np.array(mask_img.convert("L")) > 128
            else:
                mask_np = np.ones(raw.shape[:2], dtype=bool)

            # Safety fallback: If mask is empty or completely full frame (all True)
            if np.count_nonzero(mask_np) < 50 or np.count_nonzero(mask_np) == mask_np.size:
                print("Notice: Uploaded mask was empty or full-frame; isolating object automatically via rembg...")
                mask_pil = auto_segment_image(img)
                mask_np = np.array(mask_pil) > 128
        else:
            mask_pil = auto_segment_image(img)
            mask_np = np.array(mask_pil) > 128


        
        h, w = img_np.shape[:2]
        max_dim = 512
        if max(h, w) > max_dim:
            s = max_dim / max(h, w)
            nw, nh = int(w * s), int(h * s)
            img_np = np.array(Image.fromarray(img_np).resize((nw, nh), Image.Resampling.BILINEAR))
            mask_np = np.array(Image.fromarray((mask_np.astype(np.uint8) * 255)).resize((nw, nh), Image.Resampling.NEAREST)) > 0
        
        job["stage"] = "Running 3D Reconstruction"
        job["progress"] = 0.50
        with RECON_LOCK:
            torch.cuda.empty_cache()
            with torch.inference_mode():
                rgba = inf.merge_mask_to_rgba(img_np, mask_np)
                out = inf._pipeline.run(
                    rgba,
                    None,
                    seed=42,
                    stage1_only=False,
                    with_mesh_postprocess=False,
                    with_texture_baking=False,
                    with_layout_postprocess=False,
                    use_vertex_color=True,
                )
        
        job["stage"] = "Exporting 3D Models"
        job["progress"] = 0.85
        out_dir = MODELS_DIR / job_id
        out_dir.mkdir(parents=True, exist_ok=True)
        glb_path = out_dir / "model.glb"
        ply_path = out_dir / "model.ply"
        
        vertices_count = 0
        faces_count = 0
        if "glb" in out and out["glb"] is not None:
            out["glb"].export(str(glb_path))
            vertices_count = len(out["glb"].vertices)
            faces_count = len(out["glb"].faces)
        if "gs" in out and out["gs"] is not None:
            out["gs"].save_ply(str(ply_path))
        
        dt = time.time() - t0
        job["status"] = "completed"
        job["progress"] = 1.0
        job["stage"] = "Finished"
        job["completed_at"] = time.time()
        job["elapsed_sec"] = round(dt, 2)
        job["vertices"] = vertices_count
        job["faces"] = faces_count
        job["glb_size_kb"] = round(glb_path.stat().st_size / 1024, 1) if glb_path.exists() else 0
        job["ply_size_mb"] = round(ply_path.stat().st_size / (1024**2), 2) if ply_path.exists() else 0
        job["model_url"] = f"/api/models/{job_id}/model.glb"
        job["ply_url"] = f"/api/models/{job_id}/model.ply"
        print(f"Reconstruction job {job_id} ({name}) completed in {dt:.2f}s!")
    except Exception as e:
        import traceback
        traceback.print_exc()
        job["status"] = "failed"
        job["error"] = str(e)
        job["stage"] = f"Failed: {str(e)}"

@app.get("/")
@app.get("/api")
def root():
    return {
        "status": "online",
        "service": "Vis-Sam-3D Suite Server",
        "engine": "Meta AI SAM 3D Objects",
        "version": "1.0.0",
        "health": "/api/health",
        "docs": "/docs"
    }

@app.get("/api/health")
def health():
    gpus = []
    if torch.cuda.is_available():
        for i in range(torch.cuda.device_count()):
            p = torch.cuda.get_device_properties(i)
            mem = torch.cuda.memory_allocated(i) / (1024**3)
            tot = p.total_memory / (1024**3)
            gpus.append({
                "index": i,
                "name": p.name,
                "total_vram_gb": round(tot, 2),
                "allocated_vram_gb": round(mem, 2),
                "free_vram_gb": round(tot - mem, 2)
            })
    return {
        "status": "healthy",
        "device": f"cuda:0 ({gpus[0]['name']})" if gpus else "cpu",
        "cuda_available": torch.cuda.is_available(),
        "gpu_count": len(gpus),
        "gpus": gpus,
        "tunnel_url": CURRENT_TUNNEL_URL,
        "pipeline_ready": True,
        "active_jobs": sum(1 for j in JOBS.values() if j.get("status") == "processing"),
        "completed_jobs": sum(1 for j in JOBS.values() if j.get("status") == "completed"),
        "total_jobs": len(JOBS)
    }

@app.post("/api/segment")
async def segment(
    image: UploadFile = File(...),
    points_json: Optional[str] = Form(None),
    bbox_json: Optional[str] = Form(None)
):
    content = await image.read()
    img = Image.open(io.BytesIO(content)).convert("RGB")
    w, h = img.size
    pts = None
    if points_json:
        try:
            pts = json.loads(points_json)
        except Exception:
            pass
    bx = None
    if bbox_json:
        try:
            bx = json.loads(bbox_json)
        except Exception:
            pass
    mask_img = auto_segment_image(img, points=pts, bbox=bx)
    mask_id = f"mask_{uuid.uuid4().hex[:8]}"
    mask_path = MASKS_DIR / f"{mask_id}.png"
    mask_img.save(mask_path)
    buf = io.BytesIO()
    mask_img.save(buf, format="PNG")
    b64 = base64.b64encode(buf.getvalue()).decode()
    return {
        "mask_id": mask_id,
        "mask_url": f"/api/masks/{mask_id}.png",
        "mask_base64": f"data:image/png;base64,{b64}",
        "bbox": bx if bx else [0, 0, w, h],
        "width": w,
        "height": h
    }

@app.get("/api/masks/{mask_id}.png")
def get_mask_file(mask_id: str):
    p = MASKS_DIR / f"{mask_id}.png"
    if p.exists():
        return FileResponse(p, media_type="image/png")
    raise HTTPException(404, "Mask not found")

@app.post("/api/reconstruct")
async def reconstruct(
    bg: BackgroundTasks,
    image: UploadFile = File(...),
    mask: Optional[UploadFile] = File(None),
    name: str = Form("object_3d"),
    steps: int = Form(25)
):
    jid = f"job_{uuid.uuid4().hex[:8]}"
    ip = UPLOADS_DIR / f"{jid}_in.png"
    with open(ip, "wb") as f:
        f.write(await image.read())
    mp = None
    if mask:
        mp = UPLOADS_DIR / f"{jid}_mask.png"
        with open(mp, "wb") as f:
            f.write(await mask.read())
    JOBS[jid] = {
        "job_id": jid,
        "name": name,
        "status": "queued",
        "progress": 0.0,
        "stage": "Queued",
        "error": None
    }
    bg.add_task(worker, jid, ip, mp, name)
    return {"job_id": jid, "status": "queued"}

@app.get("/api/jobs/{job_id}")
def get_job(job_id: str):
    if job_id not in JOBS:
        raise HTTPException(404, "Job not found")
    return JOBS[job_id]

@app.get("/api/jobs/{job_id}/events")
@app.get("/api/reconstruct/stream/{job_id}")
async def stream_job(job_id: str):
    if job_id not in JOBS:
        raise HTTPException(404, "Job not found")
    async def event_generator():
        while True:
            job = JOBS.get(job_id)
            if not job:
                break
            yield f"data: {json.dumps(job)}" + chr(10) + chr(10)
            if job.get("status") in ["completed", "failed"]:
                break
            await asyncio.sleep(1.0)
    return StreamingResponse(event_generator(), media_type="text/event-stream")

@app.get("/api/models/{job_id}/model.glb")
@app.get("/api/models/{job_id}/{filename}")
def get_model(job_id: str, filename: str = "model.glb"):
    p = MODELS_DIR / job_id / filename
    if not p.exists():
        p = MODELS_DIR / job_id / "model.glb"
    if not p.exists():
        raise HTTPException(404, "Model file not found")
    media_type = "model/gltf-binary" if p.suffix == ".glb" else "application/octet-stream"
    return FileResponse(p, media_type=media_type, filename=p.name)

@app.get("/api/models/{job_id}/model.ply")
def get_ply(job_id: str):
    p = MODELS_DIR / job_id / "model.ply"
    if p.exists():
        return FileResponse(p, media_type="application/octet-stream", filename=f"{job_id}.ply")
    raise HTTPException(404, "PLY not found")

if __name__ == "__main__":
    uvicorn.run(app, host="0.0.0.0", port=8000, log_level="info")
'''

with open('/kaggle/working/sam3d_server.py', 'w', encoding='utf-8') as f:
    f.write(SERVER_CODE.strip())

print('✅ Step 3 Complete: Production SAM 3D Suite FastAPI server code written to /kaggle/working/sam3d_server.py!')


In [ ]:
# 4. Launch Vis-Sam-3D Server + Cloudflare HTTPS Tunnel (ALWAYS RUNNING)
import time, os, sys, subprocess, urllib.request, json

print('=' * 70)
print('  [STEP 4/4] LAUNCHING VIS-SAM-3D SUITE SERVER & CLOUDFLARE TUNNEL')
print('=' * 70)

# 1. Clean previous background instances on port 8000 (preserves external bridges)
!fuser -k 8000/tcp 2>/dev/null || true
!pkill -9 -f sam3d_server 2>/dev/null || true
time.sleep(2)

# 2. Launch FastAPI server logging to /kaggle/working/server.log
server_log = open('/kaggle/working/server.log', 'w', buffering=1)
server_proc = subprocess.Popen(
    [sys.executable, '/kaggle/working/sam3d_server.py'],
    stdout=server_log,
    stderr=subprocess.STDOUT
)
print(f"Server launched under PID {server_proc.pid}. Waiting for health check on port 8000...")

# 3. Wait and verify server is responding 200 on http://127.0.0.1:8000/api/health
server_online = False
for i in range(40):
    time.sleep(1)
    if server_proc.poll() is not None:
        print(f"❌ Server process exited prematurely with code {server_proc.returncode}!")
        break
    try:
        with urllib.request.urlopen("http://127.0.0.1:8000/api/health", timeout=1) as resp:
            if resp.status == 200:
                server_online = True
                print(f"✅ Backend server verified responding in {i+1} seconds!")
                break
    except Exception:
        pass

if not server_online:
    print("\n❌ SERVER FAILED TO START! Server log output:")
    if os.path.exists('/kaggle/working/server.log'):
        with open('/kaggle/working/server.log', 'r') as f:
            print(f.read())
    raise RuntimeError("Server failed to bind to port 8000.")

# 4. Launch Cloudflare Tunnel
try:
    from pycloudflared import try_cloudflare
except ImportError:
    !pip install -q pycloudflared
    from pycloudflared import try_cloudflare

tunnel = try_cloudflare(port=8000)
public_url = tunnel.tunnel.rstrip('/')
tunnel_url = public_url

with open('/kaggle/working/tunnel_url.txt', 'w') as f:
    f.write(public_url)

print('\n' + '=' * 70)
print('  🎉 VIS-SAM-3D SUITE SERVER IS NOW ONLINE!')
print('=' * 70)
print(f'  PUBLIC BACKEND URL : {public_url}')
print(f'  HEALTH CHECK       : {public_url}/api/health')
print(f'  LOCAL PORT         : 8000')
print('=' * 70)
print('  ➡️ Paste this URL into your Vis-Sam-3D Frontend settings:')
print(f'     {public_url}')
print('=' * 70 + '\n')

# 5. Infinite Heartbeat Loop (KEEPS NOTEBOOK RUNNING FOREVER)
import torch, psutil
print('Server active. Heartbeat monitor started (Press Stop in Kaggle to terminate):')
counter = 1
while True:
    time.sleep(30)
    ram_gb = psutil.virtual_memory().used / (1024**3)
    gpu_mb = torch.cuda.memory_allocated(0) / (1024**2) if torch.cuda.is_available() else 0
    print(f'[{time.strftime("%H:%M:%S")}] 💓 Heartbeat #{counter:04d} | RAM: {ram_gb:.2f} GB | GPU0: {gpu_mb:.1f} MB | Tunnel: {public_url}')
    counter += 1


In [ ]:
# @title 📊 Live Server Status & GPU Monitor { display-mode: "form" }
import os, urllib.request, json
SERVER_PORT = int(os.environ.get('SERVER_PORT', '8000'))
try:
    req = urllib.request.urlopen(f'http://127.0.0.1:{SERVER_PORT}/api/health', timeout=3)
    info = json.loads(req.read().decode())
    print('Status:         ONLINE (Healthy)')
    print(f'Primary Device: {info.get("device", "N/A")}')
    for g in info.get('gpus', []):
        print(f'  [GPU {g["index"]}] {g["name"]}: {g["allocated_vram_gb"]} GB / {g["total_vram_gb"]} GB VRAM')
    print(f'Active Jobs:    {info.get("active_jobs", 0)}')
    print(f'Completed Jobs: {info.get("completed_jobs", 0)}')
except Exception as e:
    print(f'Status Check: {e}')


In [ ]:
# @title 🚀 Run Self-Test Reconstruction & View 3D Model { display-mode: "form" }
import os, urllib.request, json
from IPython.display import HTML, display

active_url = "http://127.0.0.1:8000"
if os.path.exists('/kaggle/working/tunnel_url.txt'):
    try:
        with open('/kaggle/working/tunnel_url.txt', 'r') as f:
            u = f.read().strip()
            if u:
                active_url = u
    except Exception:
        pass
elif 'public_url' in locals():
    active_url = public_url

html_code = f'''
<div style="background: #0b0f19; padding: 20px; border-radius: 12px; border: 1px solid #1e293b; color: #f8fafc; font-family: sans-serif;">
  <h3 style="color: #38bdf8; margin: 0 0 10px 0;">✨ Vis-Sam-3D Suite — 3D Viewer</h3>
  <p style="color: #94a3b8; font-size: 13px;">Active Server Endpoint: <code>{active_url}</code></p>
  <script type="module" src="https://ajax.googleapis.com/ajax/libs/model-viewer/3.4.0/model-viewer.min.js"></script>
  <model-viewer style="width: 100%; height: 420px; background: #000; border-radius: 8px;" 
    src="https://raw.githubusercontent.com/KhronosGroup/glTF-Sample-Models/master/2.0/Duck/glTF-Binary/Duck.glb" 
    camera-controls auto-rotate shadow-intensity="1">
  </model-viewer>
</div>
'''
display(HTML(html_code))
